# Managing State in AutoGen

This notebook demonstrates how to save and load the state of agents, teams, and termination conditions in a multi-agent application using the AutoGen framework. This is useful for persisting agent context between runs, such as in web applications or long-running workflows.

## 1. Import Required Libraries

Import all necessary modules from autogen_agentchat, autogen_core, and autogen_ext for agent creation, messaging, and model client setup.

In [ ]:
# Import required libraries
import asyncio
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.conditions import MaxMessageTermination
from autogen_agentchat.messages import TextMessage
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.ui import Console
from autogen_core import CancellationToken
from autogen_ext.models.openai import OpenAIChatCompletionClient

## 2. Initialize and Run an AssistantAgent

Create an AssistantAgent with a system message and OpenAIChatCompletionClient. Send an initial message and display the assistant's response.

In [ ]:
# Initialize and run an AssistantAgent
async def run_and_get_response():
    model_client = OpenAIChatCompletionClient(model="gpt-4o-2024-08-06")
    assistant_agent = AssistantAgent(
        name="assistant_agent",
        system_message="You are a helpful assistant",
        model_client=model_client,
    )
    response = await assistant_agent.on_messages(
        [TextMessage(content="Write a 3 line poem on lake tangayika", source="user")], CancellationToken()
    )
    print(response.chat_message)
    await model_client.close()
    return assistant_agent, response

# Run the cell below to execute the function and see the response
# asyncio.run(run_and_get_response())

## 3. Save Agent State

Call the `save_state()` method on the AssistantAgent to retrieve its current state as a dictionary. Print the saved state.

In [ ]:
# Save the agent state after the initial conversation
async def save_agent_state():
    model_client = OpenAIChatCompletionClient(model="gpt-4o-2024-08-06")
    assistant_agent = AssistantAgent(
        name="assistant_agent",
        system_message="You are a helpful assistant",
        model_client=model_client,
    )
    await assistant_agent.on_messages(
        [TextMessage(content="Write a 3 line poem on lake tangayika", source="user")], CancellationToken()
    )
    agent_state = await assistant_agent.save_state()
    print(agent_state)
    await model_client.close()
    return agent_state

# Run the cell below to save and print the agent state
# asyncio.run(save_agent_state())

## 4. Restore Agent State

Create a new AssistantAgent instance and use the `load_state()` method to restore the previously saved state.

In [ ]:
# Restore the agent state into a new AssistantAgent instance
async def restore_agent_state(agent_state):
    model_client = OpenAIChatCompletionClient(model="gpt-4o-2024-08-06")
    new_assistant_agent = AssistantAgent(
        name="assistant_agent",
        system_message="You are a helpful assistant",
        model_client=model_client,
    )
    await new_assistant_agent.load_state(agent_state)
    await model_client.close()
    return new_assistant_agent

# Example usage:
# state = asyncio.run(save_agent_state())
# restored_agent = asyncio.run(restore_agent_state(state))

## 5. Continue Conversation After State Restore

Send a follow-up message to the restored agent and display the response, demonstrating that the conversation context is preserved.

In [ ]:
# Continue the conversation after restoring state
async def continue_conversation(agent_state):
    model_client = OpenAIChatCompletionClient(model="gpt-4o-2024-08-06")
    restored_agent = AssistantAgent(
        name="assistant_agent",
        system_message="You are a helpful assistant",
        model_client=model_client,
    )
    await restored_agent.load_state(agent_state)
    response = await restored_agent.on_messages(
        [TextMessage(content="What was the last line of the previous poem you wrote", source="user")], CancellationToken()
    )
    print(response.chat_message)
    await model_client.close()
    return response

# Example usage:
# state = asyncio.run(save_agent_state())
# asyncio.run(continue_conversation(state))